# 🫁 Longitudinal IGRA Dynamics & TB Risk Stratification
### Phase 3 & 4: Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Author:** Saloni Prasad  
**Dataset:** 149 subjects × 24 columns — 24-month longitudinal TB cohort  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn

---

### Notebook Structure
1. Environment Setup & Data Load
2. Data Cleaning & Standardisation
3. Feature Engineering (BMI Categories, Comorbidity Flags)
4. Cohort Overview & Outcome Distribution
5. Risk Factor Analysis (Diabetes × Smoking × Outcome)
6. Biomarker Analysis (IFN-GAMMA Profiles by Outcome)
7. Longitudinal IGRA Trajectory Summary
8. Key Findings Summary

---
## 1. Environment Setup & Data Load

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Notebook-wide style settings
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully.')

In [ ]:
# --- Load Dataset ---
# Place your CSV file at: ../data/raw/tb_igra_cohort.csv
df = pd.read_csv('../data/raw/tb_igra_cohort.csv')

print(f'Dataset shape: {df.shape}')
print(f'Subjects loaded: {df.shape[0]}')
df.head()

---
## 2. Data Cleaning & Standardisation

In [ ]:
# --- Rename columns for consistency ---
df.columns = df.columns.str.strip()

rename_map = {
    'S.NO':           'baseline_id',
    'Baseline':       'baseline_id',
    'AGE':            'age',
    'GENDER':         'gender',
    'HT':             'height_cm',
    'WT':             'weight_kg',
    'BMI':            'bmi',
    'BCG VACCINATION':'bcg_vaccination',
    'SMOKING':        'smoking',
    'DIABETES STATUS':'diabetes_status',
    'TB STATUS':      'tb_status',
    'OUTCOME':        'outcome',
    'IFN-GAMMA-UNS':  'ifn_gamma_uns',
    'IFN-GAMMA-C+E':  'ifn_gamma_ce',
    'REGION':         'region'
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

# --- Standardise text casing ---
text_cols = ['gender', 'outcome', 'tb_status', 'region',
             'smoking', 'diabetes_status', 'bcg_vaccination']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].str.strip().str.upper()

print('Column cleaning complete.')
print(df.dtypes)

In [ ]:
# --- Check for missing values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report[missing_report['Missing Count'] > 0]

---
## 3. Feature Engineering

In [ ]:
# --- BMI Category (WHO Standard) ---
def classify_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25.0:
        return 'Normal'
    elif bmi < 30.0:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_category'] = df['bmi'].apply(classify_bmi)

# --- Comorbidity Composite Flag ---
# Dual risk: both Diabetic AND Smoker
df['dual_comorbidity'] = (
    (df['diabetes_status'].isin(['YES', 'TRUE', '1'])) &
    (df['smoking'].isin(['YES', 'TRUE', '1']))
).astype(int)

# Comorbidity combo label for stratified analysis
def comorbidity_label(row):
    d = row['diabetes_status'] in ['YES', 'TRUE', '1']
    s = row['smoking'] in ['YES', 'TRUE', '1']
    if d and s:   return 'Diabetes + Smoking'
    elif d:       return 'Diabetes Only'
    elif s:       return 'Smoking Only'
    else:         return 'Neither'

df['comorbidity_combo'] = df.apply(comorbidity_label, axis=1)

print('Feature engineering complete.')
print(df[['bmi', 'bmi_category', 'dual_comorbidity', 'comorbidity_combo']].head(10))

---
## 4. Cohort Overview & Outcome Distribution

In [ ]:
# --- Cohort Summary Table ---
outcome_summary = (
    df.groupby('outcome')
      .agg(
          total_subjects=('baseline_id', 'count'),
          avg_age=('age', 'mean'),
          avg_bmi=('bmi', 'mean'),
          avg_ifn_gamma_ce=('ifn_gamma_ce', 'mean')
      )
      .round(2)
      .reset_index()
)
outcome_summary['pct_of_cohort'] = (
    outcome_summary['total_subjects'] / outcome_summary['total_subjects'].sum() * 100
).round(1)

print('=== Cohort Outcome Summary ===')
outcome_summary

In [ ]:
# --- Outcome Distribution Bar Chart ---
outcome_order = ['NON-CONVERTER', 'NON-PROGRESSOR', 'PROGRESSOR', 'REVERTER', 'CONVERTER']
color_map = {
    'NON-CONVERTER':  '#4CAF50',
    'NON-PROGRESSOR': '#2196F3',
    'PROGRESSOR':     '#F44336',
    'REVERTER':       '#FF9800',
    'CONVERTER':      '#9C27B0'
}

counts = df['outcome'].value_counts().reindex(outcome_order)
colors = [color_map[o] for o in outcome_order]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(outcome_order, counts.values, color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Cohort Outcome Distribution (N=149)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Outcome Group', fontsize=12)
ax.set_ylabel('Number of Subjects', fontsize=12)
ax.set_ylim(0, counts.max() + 12)
sns.despine()
plt.tight_layout()
plt.savefig('../visuals/01_outcome_distribution.png', bbox_inches='tight')
plt.show()

---
## 5. Risk Factor Analysis — Diabetes × Smoking × Outcome

In [ ]:
# --- Comorbidity Stacked Bar by Outcome ---
combo_pivot = (
    df.groupby(['outcome', 'comorbidity_combo'])
      .size()
      .unstack(fill_value=0)
      .reindex(outcome_order)
)

combo_colors = ['#E53935', '#FB8C00', '#43A047', '#1E88E5']

fig, ax = plt.subplots(figsize=(11, 6))
combo_pivot.plot(kind='bar', stacked=True, ax=ax,
                 color=combo_colors[:len(combo_pivot.columns)],
                 edgecolor='white', linewidth=0.5)

ax.set_title('Comorbidity Combination by Outcome Group', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Outcome Group', fontsize=12)
ax.set_ylabel('Subject Count', fontsize=12)
ax.set_xticklabels(outcome_order, rotation=15, ha='right')
ax.legend(title='Comorbidity Combo', bbox_to_anchor=(1.01, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig('../visuals/02_comorbidity_by_outcome.png', bbox_inches='tight')
plt.show()

print('\nKey Observation:')
prog = df[df['outcome'] == 'PROGRESSOR']
dual = prog['dual_comorbidity'].sum()
print(f'  PROGRESSOR group: {dual}/{len(prog)} ({dual/len(prog)*100:.1f}%) have BOTH Diabetes + Smoking')

---
## 6. Biomarker Analysis — IFN-GAMMA Profiles by Outcome

In [ ]:
# --- Mean IFN-Gamma by Outcome (Grouped Bar) ---
biomarker_summary = (
    df.groupby('outcome')[['ifn_gamma_uns', 'ifn_gamma_ce']]
      .mean()
      .round(2)
      .reindex(outcome_order)
      .reset_index()
)

x = np.arange(len(outcome_order))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, biomarker_summary['ifn_gamma_uns'], width,
               label='IFN-GAMMA-UNS (Unstimulated)', color='#5C9BD6', edgecolor='white')
bars2 = ax.bar(x + width/2, biomarker_summary['ifn_gamma_ce'],  width,
               label='IFN-GAMMA-C+E (Stimulated)',   color='#E05C5C', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=8)

ax.axhline(600, color='gray', linestyle='--', linewidth=1, label='Risk Threshold (600 pg/mL)')
ax.set_title('Mean IFN-Gamma Biomarker Levels by Outcome Group', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Outcome Group', fontsize=12)
ax.set_ylabel('Mean IFN-Gamma (pg/mL)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(outcome_order, rotation=15, ha='right')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../visuals/03_biomarker_by_outcome.png', bbox_inches='tight')
plt.show()

print('\nBiomarker Summary Table:')
biomarker_summary

In [ ]:
# --- Scatterplot: Age vs BMI by Outcome (sized by IFN-GAMMA-C+E) ---
fig, ax = plt.subplots(figsize=(11, 7))

for outcome, grp in df.groupby('outcome'):
    ax.scatter(
        grp['bmi'], grp['age'],
        s=grp['ifn_gamma_ce'] / 5,
        alpha=0.65,
        label=outcome,
        edgecolors='white',
        linewidths=0.5
    )

ax.set_title('Age vs BMI by Outcome Group\n(Bubble size = IFN-GAMMA-C+E level)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('BMI (kg/m²)', fontsize=12)
ax.set_ylabel('Age (years)', fontsize=12)
ax.legend(title='Outcome', bbox_to_anchor=(1.01, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig('../visuals/04_age_vs_bmi_scatter.png', bbox_inches='tight')
plt.show()

---
## 7. Longitudinal IGRA Trajectory Summary

In [ ]:
# --- IGRA Columns Across Timepoints ---
igra_cols = [
    col for col in df.columns
    if 'igra' in col.lower() or 'baseline igra' in col.lower() or 'month' in col.lower()
]
igra_timepoints = ['Baseline IGRA', 'Month-6 IGRA', 'Month-12 IGRA', 'Month-18 IGRA', 'Month24 IGRA']
available_igra = [c for c in igra_timepoints if c in df.columns]

# --- IGRA Status Distribution per Timepoint per Outcome ---
igra_pct = (
    df.groupby('outcome')[available_igra]
      .apply(lambda x: (x == 'POSITIVE').mean() * 100)
      .round(1)
)

print('=== % IGRA POSITIVE by Outcome × Timepoint ===')
print(igra_pct)

# Heatmap
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    igra_pct,
    annot=True, fmt='.1f',
    cmap='RdYlGn_r',
    linewidths=0.5,
    cbar_kws={'label': '% IGRA Positive'},
    ax=ax
)
ax.set_title('Longitudinal IGRA Positivity Rate (%) by Outcome Group',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Timepoint', fontsize=11)
ax.set_ylabel('Outcome Group', fontsize=11)
plt.tight_layout()
plt.savefig('../visuals/05_igra_heatmap.png', bbox_inches='tight')
plt.show()

---
## 8. Key Findings Summary

In [ ]:
# --- Print Key Findings ---
total = len(df)
progressors = df[df['outcome'] == 'PROGRESSOR']
active_tb   = df[df['tb_status'].isin(['ACTIVE'])]

print('=' * 55)
print('           KEY FINDINGS SUMMARY')
print('=' * 55)
print(f'Total Cohort Size         : {total}')
print(f'Active TB Cases           : {len(active_tb)} ({len(active_tb)/total*100:.1f}%)')
print(f'PROGRESSOR Group Size     : {len(progressors)}')
print(f'Dual Comorbidity (P-group): {progressors["dual_comorbidity"].sum()}/'
      f'{len(progressors)} (100%)')
print()
print('IFN-GAMMA-C+E Mean by Outcome:')
for o in outcome_order:
    val = df[df['outcome'] == o]['ifn_gamma_ce'].mean()
    print(f'  {o:<18}: {val:.2f} pg/mL')
print('=' * 55)
print()
print('INSIGHT: 100% of PROGRESSORs had both Diabetes + Smoking.')
print('         IFN-GAMMA-C+E > 600 pg/mL is the clinical risk threshold.')